In [1]:
# Initial setup

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.0f}".format

In [2]:
# The data cleaning process in the previous step.
# And data pipeline from the previous step
# We put them into a single function.

In [3]:
def ETL_pipeline(year:int):
    df = pd.read_csv(f"./data/raw/generation_tech_{year}.csv")
    # Clean up generation values
    df['Generation (MW)'] = pd.to_numeric(df['Generation (MW)'], errors='coerce')
    # Extract date from MTU
    df['start_time'] = df['MTU (CET/CEST)'].str.split(' - ').str[0]
    df['date'] = df['start_time'].str.split(' ').str[0]
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df = df[['date','Production Type','Generation (MW)']]
    df = df.rename(columns={
        'Production Type':'tech',
        'Generation (MW)':'gen',
    })
    df['daily_electricity_by_gen'] = (
        df.groupby(['date', 'tech'])['gen']
        .transform('sum')
    )
    df['daily_electricity'] = (
        df.groupby(['date'])['gen']
        .transform('sum')
    )
    df['daily_avg_share_gen'] = df['daily_electricity_by_gen'] / df['daily_electricity']
    g = df.groupby(['date','tech']).agg(
        avg_gen=('gen','mean'),
        std_gen=('gen','std'),
    ).reset_index()
    g['cov'] = g['std_gen'] / g['avg_gen'] # This will be a percentage in Power BI
    g2 = df.groupby(['date','tech']).agg(
        daily_electricity_by_gen=('daily_electricity_by_gen','mean'),
        daily_avg_share_gen=('daily_avg_share_gen','mean')
    ).reset_index()
    m = pd.merge(g,g2,'left',['date','tech'])
    g3 = df.groupby(['date']).agg(daily_electricity=('daily_electricity','mean')).reset_index()
    
    return m,g3

In [4]:
m,g3 = ETL_pipeline(year=2015)

In [5]:
m.head()

,date,tech,avg_gen,std_gen,cov,daily_electricity_by_gen,daily_avg_share_gen
0,2015-01-01,Biomass,201,6,0,"4,813",0
1,2015-01-01,Energy storage,NaN,NaN,NaN,0,0
2,2015-01-01,Fossil Brown coal/Lignite,NaN,NaN,NaN,0,0
3,2015-01-01,Fossil Coal-derived gas,NaN,NaN,NaN,0,0
4,2015-01-01,Fossil Gas,"2,425",148,0,"58,190",0


In [6]:
g3.head()

,date,daily_electricity
0,2015-01-01,"1,725,680"
1,2015-01-02,"1,793,290"
2,2015-01-03,"1,747,153"
3,2015-01-04,"1,643,712"
4,2015-01-05,"1,841,655"


In [7]:
# Loop over all raw datasets from 2015 - 2024
# WHY until 2024? because in 2025, the MTU changed from 1H to 15min
# The logic to calculate the Daily Electricity Production (MWh) needs to be modified
# Create two "compilations"
# Run the ETL pipeline for 2015 - 2024 and incrementally add each year to the compilations
# Run the modified ETL pipeline for 2025 and do the same.

In [8]:
daily_electricity = pd.DataFrame()
tech_stats = pd.DataFrame()

for year in range(2015,2025):
    m,g3 = ETL_pipeline(year)
    daily_electricity = pd.concat([daily_electricity,g3])
    tech_stats = pd.concat([tech_stats,m])

In [9]:
daily_electricity

,date,daily_electricity
0,2015-01-01,"1,725,680"
1,2015-01-02,"1,793,290"
2,2015-01-03,"1,747,153"
3,2015-01-04,"1,643,712"
4,2015-01-05,"1,841,655"
...,...,...
361,2024-12-27,"1,747,820"
362,2024-12-28,"1,720,095"
363,2024-12-29,"1,587,743"
364,2024-12-30,"1,583,550"


In [10]:
tech_stats

,date,tech,avg_gen,std_gen,cov,daily_electricity_by_gen,daily_avg_share_gen
0,2015-01-01,Biomass,201,6,0,"4,813",0
1,2015-01-01,Energy storage,NaN,NaN,NaN,0,0
2,2015-01-01,Fossil Brown coal/Lignite,NaN,NaN,NaN,0,0
3,2015-01-01,Fossil Coal-derived gas,NaN,NaN,NaN,0,0
4,2015-01-01,Fossil Gas,"2,425",148,0,"58,190",0
...,...,...,...,...,...,...,...
7681,2024-12-31,Other renewable,NaN,NaN,NaN,0,0
7682,2024-12-31,Solar,"1,171","1,855",2,"28,095",0
7683,2024-12-31,Waste,298,4,0,"7,146",0
7684,2024-12-31,Wind Offshore,"1,026",393,0,"24,623",0


In [11]:
# Starting 2025, the Market Time Unit increases resolution from 1 Hour to 15 Minutes.

def Modified_ETL_pipeline(year:int = 2025):
    # SAME DATA CLEANUP PROCEDURE
    df = pd.read_csv(f"./data/raw/generation_tech_{year}.csv")
    # clean up generation values
    df['Generation (MW)'] = pd.to_numeric(df['Generation (MW)'], errors='coerce')
    # extract date from MTU
    df['start_time'] = df['MTU (CET/CEST)'].str.split(' - ').str[0]
    df['date'] = df['start_time'].str.split(' ').str[0]
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df = df[['date','Production Type','Generation (MW)']]
    
    # WHAT CHANGED
    df = df.rename(columns={
        'Production Type':'tech',
        'Generation (MW)':'power_15_min', # remember energy is generated for 15 mins at this rate
    })
    # energy produced in 15 mins = Power output (MW) * 0.25 hours
    df['energy_15_min'] = df['power_15_min'] / 4 
    # after scaling correctly, we can sum up all the 15-min energy values per tech for the same day.
    # to get the daily energy produced
    df['daily_electricity_by_gen'] = (
        df.groupby(['date', 'tech'])['energy_15_min']
        .transform('sum')
    )
    df['daily_electricity'] = (
        df.groupby(['date'])['energy_15_min']
        .transform('sum')
    )
    
    # SAME DATA AGGREGATION PROCEDURE
    df['daily_avg_share_gen'] = df['daily_electricity_by_gen'] / df['daily_electricity']
    g = df.groupby(['date','tech']).agg(
        avg_gen=('power_15_min','mean'),
        std_gen=('power_15_min','std'),
    ).reset_index()
    g['cov'] = g['std_gen'] / g['avg_gen'] # This will be a percentage in Power BI
    g2 = df.groupby(['date','tech']).agg(
        daily_electricity_by_gen=('daily_electricity_by_gen','mean'),
        daily_avg_share_gen=('daily_avg_share_gen','mean')
    ).reset_index()
    m = pd.merge(g,g2,'left',['date','tech'])
    g3 = df.groupby(['date']).agg(daily_electricity=('daily_electricity','mean')).reset_index()
    return m,g3

In [12]:
m,g3 = Modified_ETL_pipeline(2025)
daily_electricity = pd.concat([daily_electricity,g3])
tech_stats = pd.concat([tech_stats,m])

In [13]:
tech_stats.to_csv('./data/tech_stats.csv', index=False)

In [14]:
daily_electricity.to_csv('./data/daily_electricity.csv', index=False)